# Module 7: Sandboxes & Databricks

> Part of the **Modular Workshops** series. Standalone, ~30 min.

Two production concerns for the in-store shopping assistant, each about *where work runs and where data lives*:

1. **Sandboxes** — hand the agent a real execution environment so the parts of a shopping task that must be **exact and auditable** (basket totals, budget checks, list validation) are produced by *code the agent runs*, not by the model doing mental arithmetic.
2. **Databricks** — connect the workshop to a Databricks **Unity Catalog** lakehouse two ways:
   - **as an eval-data source** — pull your labeled eval cases from a Delta table into LangSmith (Part 2), and
   - **as a runtime capability** — give the agent a **tool** that queries live inventory/pricing from Databricks while it helps a shopper (Part 3), including the Unity Catalog **per-user vs. global credentials** decision.

Everything degrades gracefully: the sandbox cells are optional, and the Databricks cells fall back to the local store catalog when `DATABRICKS_*` env vars are unset — so the notebook runs top-to-bottom offline.

## Setup

In [ ]:
import sys, os, json
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(dotenv_path=project_root / ".env", override=True)

import re
from langsmith import Client, uuid7
from utils.models import model

client = Client()


def qualify(name: str) -> str:
    """Namespace a shared resource name per user so parallel workshop runs
    don't collide on datasets / snapshots / Context Hub repos."""
    who = (os.environ.get("USER") or os.environ.get("USERNAME") or "workshop")
    slug = re.sub(r"[^a-z0-9-]+", "-", who.lower()).strip("-") or "workshop"
    return f"{name}-{slug}"


# The shopping assistant's eval cases — the same shape the LangSmith module uses
# (a query, a reference answer, and the expected tool trajectory). Part 2 can
# replace this list with rows pulled from Databricks; the shape stays identical.
examples = [
    {"inputs": {"query": "What aisle is the milk in, and is it in stock?"},
     "outputs": {"reference_answer": "Milk is in aisle D2 (Dairy) and is in stock.",
                 "trajectory": ["store_directory"]}},
    {"inputs": {"query": "Is ground beef available? If not, what should I grab instead?"},
     "outputs": {"reference_answer": "Ground beef is out of stock; suggest a substitution such as ground turkey.",
                 "trajectory": ["check_stock", "find_substitution"]}},
    {"inputs": {"query": "I need salsa and tomatoes — give me the aisles and prices."},
     "outputs": {"reference_answer": "Salsa: aisle 6, $2.79. Tomatoes: aisle 1 (Produce), $0.99.",
                 "trajectory": ["store_directory", "store_directory"]}},
    {"inputs": {"query": "Plan my taco run: tortillas, ground beef, salsa. Build an aisle-ordered route and flag anything out of stock."},
     "outputs": {"reference_answer": "An aisle-ordered route for tortillas, ground beef (flagged out of stock), and salsa.",
                 "trajectory": ["write_todos", "store_directory", "store_directory", "store_directory"]}},
]
print(f"Setup ready — {len(examples)} local eval examples.")

---
# Part 1. Give the agent a sandbox for deterministic shopping math

> **Optional cells — skip them and the rest of the notebook still runs.** They need `langsmith[sandbox]` installed and a workspace with the Sandbox feature enabled. Run `uv add "langsmith[sandbox]"` if the import fails.

The shopping assistant answers from the model, the store tools, and web search. But it shouldn't *reason* its way through the parts of a shopping trip that must be **exact and auditable** — basket totals, per-unit vs. pack pricing, a budget cap, "does this list actually fit the $50 the shopper gave me?". LLMs are unreliable at arithmetic and easy to talk out of a rule; a sandbox isn't.

So we hand the agent a **LangSmith sandbox** as its execution backend. Now it can *write and run real Python* to total a basket, apply a coupon, and check it against the budget — deterministic, reproducible, and (because tracing is on) every sandbox command shows up in the trace next to the model's reasoning. That's the split you want in production: the model decides *what* to compute and *how to explain it*; the sandbox produces the *ground-truth numbers and pass/fail*.

We do it in **two cells** so a slow image build doesn't collide with the agent run:

1. **Build the snapshot** (below) — compiles the Docker image into a bootable snapshot. It's the slow step, so we give it a generous timeout and wait until it reports `ready`. Re-running is cheap once the snapshot exists.
2. **Boot a sandbox from the ready snapshot and run the agent** (next cell) — fast, and safe to re-run on its own.

Concretely, we ask the agent to price a taco-night basket, apply a 15% loyalty discount, and tell the shopper whether it fits a **$25 budget** — by executing code, not by guessing.

In [ ]:
# OPTIONAL — Cell 1 of 2: build the snapshot and wait until it is ready.
# Requires `langsmith[sandbox]`; run `uv add "langsmith[sandbox]"` if the import fails.
from langsmith.sandbox import SandboxClient

sandbox_client = SandboxClient()  # uses LANGSMITH_ENDPOINT + LANGSMITH_API_KEY
snapshot_name = qualify("basket-math")

# `create_snapshot` blocks until the image build finishes. Building can take a
# minute or two the first time, so bump the default 60s timeout. `fs_capacity_bytes`
# is REQUIRED and is a real filesystem size in *bytes* — 2 GiB here (not 512 bytes,
# which is far too small to unpack a Python image and is what makes the build fail). 
# The maximum amount of disk space is 60gb
snapshot = sandbox_client.create_snapshot(
    name=snapshot_name,
    docker_image="python:3.12-slim",
    fs_capacity_bytes=2 * 1024**3,  # 2 GiB
    timeout=300,
)

# Defensive: create_snapshot already waits, but poll once more so this cell only
# succeeds on a genuinely ready snapshot (raises on 'failed' with the build reason).
snapshot = sandbox_client.wait_for_snapshot(snapshot.id, timeout=300)
print(f"Snapshot {snapshot.name!r} is {snapshot.status} (id={snapshot.id}).")

In [ ]:
# OPTIONAL — Cell 2 of 2: boot a sandbox from the ready snapshot and run the agent.
# Safe to re-run on its own; it reuses `sandbox_client` / `snapshot` from the cell above.
from deepagents.backends import LangSmithSandbox
from deepagents import create_deep_agent

# A tiny synthetic basket to price. In production the prices would come from the
# store's pricing system (see Part 3: the agent can pull them live from Databricks).
basket_json = json.dumps({
    "shopper": "loyalty member #4471",
    "budget_usd": 25.00,
    "loyalty_discount_pct": 15,
    "lines": [
        {"item": "ground beef",     "unit_price": 6.99, "qty": 1},
        {"item": "tortillas",       "unit_price": 2.49, "qty": 2},
        {"item": "shredded cheese", "unit_price": 3.99, "qty": 1},
        {"item": "salsa",           "unit_price": 2.79, "qty": 2},
        {"item": "avocado",         "unit_price": 1.19, "qty": 3},
    ],
})

with sandbox_client.sandbox(snapshot_id=snapshot.id, timeout=120) as sb:
    # Wrap the live sandbox as a deepagents backend so the agent's file + shell
    # tools run *inside* it instead of on the notebook host.
    sandbox_agent = create_deep_agent(
        model=model,
        tools=[],
        system_prompt=(
            "You are an in-store shopping assistant with a Python sandbox. "
            "For anything that must be exact -- basket subtotal, applying the loyalty "
            "discount, and the budget check -- WRITE AND RUN PYTHON. Never do the "
            "arithmetic in your head. Report a clear line-item breakdown, the discounted "
            "total, and a PASS/FAIL on the budget, then a short plain-language summary."
        ),
        backend=LangSmithSandbox(sb),
    )

    cfg = {"configurable": {"thread_id": str(uuid7())}}
    result = sandbox_agent.invoke(
        {"messages": [{"role": "user", "content":
            "Total this basket by executing code, apply the loyalty discount, and tell me "
            f"whether it fits my budget. Show the math.\n\n{basket_json}"
        }]},
        config=cfg,
    )
    print(result["messages"][-1].text)

# Why this matters: the subtotal (6.99 + 2*2.49 + 3.99 + 2*2.79 + 3*1.19 = 24.06),
# the 15% discount, and the budget PASS/FAIL are produced by code the agent RAN --
# not by the model asserting them. Model + each sandbox command is one trace you
# can inspect, evaluate, and route to review.

---
# Part 2. Databricks as the eval-data source of truth

If your labeled eval cases already live in Databricks, keep them there and pull from them. The flow is: **read creds → (optionally) seed a demo table → query the Delta table → map rows into the LangSmith example shape**. The result reassigns `examples`, so the dataset-creation cell at the end is unchanged whether the data came from Databricks or the local list.

**Interoperability, not migration.** Databricks stays the system of record for the rows (governed by **Unity Catalog**, versioned as a Delta table). LangSmith is where you run experiments, judge outputs, and observe traces. Re-pull on every run so the LangSmith dataset is always a snapshot of the current Delta table.

**Setup.** Add these to your `.env` (a Databricks **Free Edition** workspace works). Leave them unset to skip this section and use the local `examples`.

```bash
DATABRICKS_HOST="dbc-xxxxxxxx-xxxx.cloud.databricks.com"    # no https://
DATABRICKS_HTTP_PATH="/sql/1.0/warehouses/xxxxxxxxxxxxxxxx" # SQL warehouse
DATABRICKS_TOKEN="dapi..."                                  # personal access token
# Optional — where the tables live (defaults shown):
DATABRICKS_CATALOG="workspace"
DATABRICKS_SCHEMA="default"
DATABRICKS_EVAL_TABLE="shopping_agent_evals"
```

Install the connector once: `uv pip install databricks-sql-connector`.

In [ ]:
# --- Databricks config (read from .env; graceful skip if unset) ---
DATABRICKS_HOST = os.environ.get("DATABRICKS_HOST")
DATABRICKS_HTTP_PATH = os.environ.get("DATABRICKS_HTTP_PATH")
DATABRICKS_TOKEN = os.environ.get("DATABRICKS_TOKEN")

DBX_CATALOG = os.environ.get("DATABRICKS_CATALOG", "workspace")
DBX_SCHEMA = os.environ.get("DATABRICKS_SCHEMA", "default")
DBX_EVAL_TABLE = os.environ.get("DATABRICKS_EVAL_TABLE", "shopping_agent_evals")

# Validate identifiers before they ever go near a SQL string (never trust these
# blindly even from env). Unity Catalog names are dotted identifiers.
_IDENT = re.compile(r"^[A-Za-z0-9_]+$")

def _fqn(catalog, schema, table):
    for part in (catalog, schema, table):
        if not _IDENT.match(part):
            raise ValueError(f"Unsafe SQL identifier: {part!r}")
    print(f"{catalog}.{schema}.{table}")
    return f"{catalog}.{schema}.{table}"

DBX_EVAL_FQN = _fqn(DBX_CATALOG, DBX_SCHEMA, DBX_EVAL_TABLE)
databricks_configured = all([DATABRICKS_HOST, DATABRICKS_HTTP_PATH, DATABRICKS_TOKEN])


def databricks_connection(access_token: str | None = None):
    """Open a Databricks SQL warehouse connection. Requires databricks-sql-connector.

    `access_token` lets a caller pass a per-user token (see Part 3's Unity Catalog
    note); it defaults to the workspace/service token from the environment.
    """
    from databricks import sql  # pip install databricks-sql-connector
    return sql.connect(
        server_hostname=DATABRICKS_HOST,
        http_path=DATABRICKS_HTTP_PATH,
        access_token=access_token or DATABRICKS_TOKEN,
    )


if databricks_configured:
    print("Databricks configured. Eval table:", DBX_EVAL_FQN)
else:
    print("Databricks not configured (DATABRICKS_* unset) — Part 2 will use local `examples`.")

### 2.1 (Optional) Seed a demo eval table in your workspace
Skip this if your team already maintains the source-of-truth table. The schema maps straight into LangSmith examples: one row per eval case, with the tool trajectory stored as a JSON string (portable across engines).

In [ ]:
if databricks_configured:
    seed_rows = [
        (e["inputs"]["query"], e["outputs"]["reference_answer"],
         json.dumps(e["outputs"]["trajectory"]))
        for e in examples  # reuse the local examples as seed data
    ]
    with databricks_connection() as conn, conn.cursor() as cur:
        cur.execute(f"CREATE SCHEMA IF NOT EXISTS {DBX_CATALOG}.{DBX_SCHEMA}")
        cur.execute(f"""
            CREATE TABLE IF NOT EXISTS {DBX_EVAL_FQN} (
                query STRING,
                reference_answer STRING,
                trajectory STRING  -- JSON array of tool names
            )
        """)
        # Idempotent reseed: clear then insert the demo rows.
        cur.execute(f"DELETE FROM {DBX_EVAL_FQN}")
        cur.executemany(
            f"INSERT INTO {DBX_EVAL_FQN} (query, reference_answer, trajectory) VALUES (?, ?, ?)",
            seed_rows,
        )
    print(f"Seeded {len(seed_rows)} rows into {DBX_EVAL_FQN}")
else:
    print("Skipped seeding — Databricks not configured.")

### 2.2 (Optional) Pull from Databricks into the LangSmith example shape
Reassigns `examples` so the dataset-creation cell below is identical whether the data came from Databricks or the local default.

In [ ]:
def load_examples_from_databricks() -> list[dict]:
    with databricks_connection() as conn, conn.cursor() as cur:
        cur.execute(f"SELECT query, reference_answer, trajectory FROM {DBX_EVAL_FQN}")
        rows = cur.fetchall()

    mapped = []
    for query, reference_answer, trajectory in rows:
        # `trajectory` is a JSON array string in the table; parse to a list.
        traj = json.loads(trajectory) if isinstance(trajectory, str) else list(trajectory or [])
        mapped.append({
            "inputs": {"query": query},
            "outputs": {"reference_answer": reference_answer, "trajectory": traj},
        })
    return mapped


if databricks_configured:
    examples = load_examples_from_databricks()
    print(f"Pulled {len(examples)} examples from {DBX_EVAL_FQN}")
    for e in examples[:2]:
        print(" -", e["inputs"]["query"][:70], "->", e["outputs"]["trajectory"])
else:
    print(f"Using {len(examples)} local examples (Databricks not configured).")

In [ ]:
dataset_name = qualify("shopping-agent-evals")

if client.has_dataset(dataset_name=dataset_name):
    existing = client.read_dataset(dataset_name=dataset_name)
    client.delete_dataset(dataset_id=existing.id)
    print(f"Deleted existing dataset '{dataset_name}'")

dataset = client.create_dataset(dataset_name)
client.create_examples(
    inputs=[e["inputs"] for e in examples],
    outputs=[e["outputs"] for e in examples],
    dataset_id=dataset.id,
)
print(f"Created dataset '{dataset_name}' with {len(examples)} examples")
print(f"View: {dataset.url}")

---
# Part 3. Databricks as a runtime capability — Unity Catalog functions as tools

Part 2 used Databricks to *build eval data*. Part 3 is different: the agent itself **queries Databricks while it's helping a shopper**. The LangChain-native way to do this is **not** hand-written SQL from Python — it's the **Unity Catalog AI toolkit**. You register the lookup as a **Unity Catalog function** (a governed SQL/Python UDF that lives in UC), then load it as agent tools with `UCFunctionToolkit`. The agent calls the *function*; Unity Catalog governs it.

Why this is the right pattern:
- **Governance lives in UC, not your app.** The function is a first-class UC object with grants, lineage, and audit. Access control is enforced by Unity Catalog per principal — see §3.2.
- **One definition, many callers.** The same UC function is reusable across agents, notebooks, and dashboards — not re-implemented per app.
- **No SQL string-building in the agent.** The toolkit generates a typed tool from the function's signature; arguments are passed as typed parameters, so there's no injection surface in your code.

Install once: `pip install unitycatalog-langchain[databricks]` (brings in `databricks-langchain`). When Unity Catalog isn't configured, the cells below **fall back to a local tool over the mock catalog**, so the notebook still runs offline.

The flow: **(1)** create a UC function `inventory_lookup(item)` that returns aisle/stock/price → **(2)** load it as a tool via `UCFunctionToolkit` → **(3)** bind it to the shopping agent.

In [ ]:
# --- Step 1: seed a store_inventory Delta table, then register a Unity Catalog
# SQL FUNCTION that reads it. Governance (grants, audit, lineage) applies to the
# UC function, and any agent/dashboard can reuse it. Guarded so the notebook
# runs offline.
#
# IMPORTANT: this is a SQL UDF, not a Python UDF. A UC *Python* UDF runs in a
# sandbox with no Spark session and no ambient credentials, so it CANNOT query
# another table (you'd hit "cannot configure default credentials"). A SQL UDF
# executes as SQL on the warehouse and can read the table directly.
from agents.research_agent import STORE_CATALOG

DBX_INV_TABLE = os.environ.get("DATABRICKS_INVENTORY_TABLE", "store_inventory")
DBX_INV_FQN = _fqn(DBX_CATALOG, DBX_SCHEMA, DBX_INV_TABLE)
DBX_INV_FUNC = os.environ.get("DATABRICKS_INVENTORY_FUNCTION", "inventory_lookup")
UC_FUNC_FQN = _fqn(DBX_CATALOG, DBX_SCHEMA, DBX_INV_FUNC)

uc_client = None
uc_ready = False
if databricks_configured:
    try:
        rows = [(k, v["aisle"], bool(v["in_stock"]), float(v["price"]))
                for k, v in STORE_CATALOG.items()]
        with databricks_connection() as conn, conn.cursor() as cur:
            cur.execute(f"CREATE SCHEMA IF NOT EXISTS {DBX_CATALOG}.{DBX_SCHEMA}")
            cur.execute(f"""
                CREATE TABLE IF NOT EXISTS {DBX_INV_FQN} (
                    item STRING, aisle STRING, in_stock BOOLEAN, price DOUBLE
                )
            """)
            cur.execute(f"DELETE FROM {DBX_INV_FQN}")
            cur.executemany(
                f"INSERT INTO {DBX_INV_FQN} (item, aisle, in_stock, price) VALUES (?, ?, ?, ?)",
                rows,
            )
            print(f"Seeded {len(rows)} rows into {DBX_INV_FQN}")

            # Drop any pre-existing function first. CREATE OR REPLACE will
            # NOT convert a non-SQL (e.g. Python) function into a SQL one --
            # it errors -- so an old registration from a prior run must be
            # removed before we (re)create the SQL version.
            cur.execute(f"DROP FUNCTION IF EXISTS {UC_FUNC_FQN}")

            # SQL UDF. The parameter is named `lookup_item` (distinct from the
            # table column `item`) so the WHERE clause is unambiguous. Returns a
            # single formatted line, or a "not found" message via COALESCE.
            cur.execute(f"""
                CREATE OR REPLACE FUNCTION {UC_FUNC_FQN}(lookup_item STRING)
                RETURNS STRING
                COMMENT 'Look up a grocery item aisle, live stock status, and price.'
                RETURN COALESCE(
                    (SELECT concat(
                        t.item, ': aisle ', t.aisle, ', ',
                        CASE WHEN t.in_stock THEN 'in stock' ELSE 'OUT OF STOCK' END,
                        ', $', format_number(t.price, 2))
                     FROM {DBX_INV_FQN} AS t
                     WHERE lower(t.item) = lower(lookup_item)
                     LIMIT 1),
                    concat(lookup_item, ' not found in store inventory.')
                )
            """)
            print("Registered UC SQL function:", UC_FUNC_FQN)

        # The toolkit will load this already-registered UC function by name.
        from unitycatalog.ai.core.databricks import DatabricksFunctionClient
        uc_client = DatabricksFunctionClient()
        uc_ready = True
    except Exception as e:
        print(f"Could not register the UC function ({type(e).__name__}: {e}).")
        print("Falling back to the local tool below.")
else:
    print("Databricks not configured — Part 3 will use a local fallback tool.")

In [ ]:
# --- Step 2: load the UC function as agent tools (or fall back locally) ---
from langchain_core.tools import tool
from agents.research_agent import STORE_CATALOG   # local fallback data

def _local_inventory_tool():
    """Offline fallback: same interface as the UC function, reads the mock catalog.

    The argument is named `lookup_item` to match the UC SQL function's parameter
    (which is `lookup_item`, not `item`, so it doesn't collide with the table's
    `item` column). Keeping the names identical means every call site works
    whether the tool came from the Unity Catalog toolkit or this fallback.
    """
    @tool(parse_docstring=True)
    def inventory_lookup(lookup_item: str) -> str:
        """Look up an item's aisle, live stock status, and price from store inventory.

        Args:
            lookup_item: The grocery item to look up (e.g. "milk", "ground beef").
        """
        entry = STORE_CATALOG.get(lookup_item.strip().lower())
        if entry is None:
            return f"'{lookup_item}' not found in store inventory."
        status = "in stock" if entry["in_stock"] else "OUT OF STOCK"
        return f"{lookup_item}: aisle {entry['aisle']}, {status}, ${float(entry['price']):.2f}"
    return [inventory_lookup]

if uc_ready:
    # The LangChain-native path: turn the governed UC function into agent tools.
    from unitycatalog.ai.langchain.toolkit import UCFunctionToolkit
    toolkit = UCFunctionToolkit(function_names=[UC_FUNC_FQN], client=uc_client)
    inventory_tools = toolkit.tools
    print("Loaded UC tools:", [t.name for t in inventory_tools])
else:
    inventory_tools = _local_inventory_tool()
    print("Using local fallback tool:", [t.name for t in inventory_tools])

# Quick check. Use the tool's own first argument name so this works regardless of
# whether the tool is the UC one (arg: lookup_item) or the local fallback.
_tool = inventory_tools[0]
_arg = next(iter(_tool.args.keys()))
print(_tool.invoke({_arg: "salsa"}))

### 3.1 Wire the UC tools into the shopping agent

`inventory_tools` is a normal list of LangChain tools — whether it came from the Unity Catalog toolkit or the local fallback. Add it to the agent's toolset and the model calls it like any other tool. Now the assistant answers stock/price from a **governed UC function** (or the local fallback), and every call lands in the trace.

In [ ]:
from deepagents import create_deep_agent

inventory_agent = create_deep_agent(
    model=model,
    tools=inventory_tools,   # from the UC toolkit (or the local fallback)
    system_prompt=(
        "You are an in-store shopping assistant. Use the inventory_lookup tool for "
        "authoritative aisle, stock, and price — it reads live store inventory from "
        "Unity Catalog. Never guess a price or whether something is in stock; look it up."
    ),
)

cfg = {"configurable": {"thread_id": str(uuid7())}}
res = inventory_agent.invoke(
    {"messages": [{"role": "user", "content":
        "I'm making tacos — where's the salsa, is ground beef in stock, and what do they cost?"
    }]},
    config=cfg,
)
print(res["messages"][-1].text)

### 3.2 Unity Catalog credentials: per-user vs. global (service principal)

The agent now reads a Unity Catalog table, so the real governance question is **whose identity the query runs as**. Unity Catalog enforces table/row/column grants *per principal*, so the credential you hand the agent decides what it can see.

**Global / service-principal credential (one identity for the agent).**
- The agent connects as a single service principal (or a shared PAT). Every shopper's request queries with the *same* grants.
- **Use when:** the data is the same for everyone (a store's public inventory, aisle, and shelf price — exactly our `store_inventory` case), or the app enforces its own row filtering on top.
- **Pros:** simple, one connection to pool and cache; easy to reason about; no per-user token plumbing.
- **Cons:** no UC-level per-user isolation — if a shopper should only see *their* data, a global identity can't enforce that; and the audit log shows the service principal, not the end user.

**Per-user credential (on-behalf-of / OBO — the agent queries as the end user).**
- Each request carries the *shopper's* (or employee's) identity — an OBO token or user-scoped credential — so Unity Catalog applies **that user's** grants, row filters, and column masks.
- **Use when:** results must be scoped to the person — loyalty history, saved carts, employee-only cost/margin columns, store-manager dashboards. This is the honest answer for anything sensitive or personalized.
- **Pros:** UC does the access control and the audit trail names the real user; least-privilege by construction.
- **Cons:** you must propagate the user identity through the agent to the tool (token exchange / OBO), can't share one pooled connection across users, and unauthenticated/background runs (crons, evals) have no user to act as.

**Rule of thumb for this agent:** public catalog data (aisle/stock/shelf price) → **global service principal**. Anything user-specific (a shopper's history, an associate's cost/margin view) → **per-user / OBO**, so Unity Catalog — not your prompt — enforces who sees what.

Because the lookup is a **Unity Catalog function**, the identity question is about which principal the `DatabricksFunctionClient` authenticates as when the tool executes. Configure a **service principal** for public catalog data, or exchange the end user's identity (**OBO**) so UC evaluates *their* grants for personalized/sensitive functions. Where the identity comes from depends on your deployment — e.g. `configurable` on the run, or the deployment's auth context — but the enforcement is UC's, not your prompt's.

---
## Recap

| Concern | Pattern | Why |
|---|---|---|
| **Exact, auditable math** | LangSmith **sandbox** as the agent's backend; model writes & runs Python | Basket totals / budget checks are code the agent ran, in the trace — not model arithmetic |
| **Eval data lives in Databricks** | Pull a Delta table → map to LangSmith examples (Part 2) | Unity Catalog stays system-of-record; LangSmith runs the experiments |
| **Live data as a capability** | **Unity Catalog function** loaded via `UCFunctionToolkit` (Part 3) | The agent calls a governed UC function for stock/price; falls back to local offline |
| **Who the query runs as** | UC **global service principal** for public data; **per-user / OBO** for personalized/sensitive data | Unity Catalog enforces access per principal — pick the identity accordingly |

Everything falls back gracefully: the sandbox cells are optional, and the Databricks cells use the local store catalog when `DATABRICKS_*` is unset.

**Docs:** [LangSmith Sandboxes](https://docs.langchain.com/langsmith/sandboxes) · [Unity Catalog toolkit for LangChain](https://docs.unitycatalog.io/ai/integrations/langchain/) · [Unity Catalog](https://docs.databricks.com/data-governance/unity-catalog/index.html) · [On-behalf-of auth](https://docs.databricks.com/dev-tools/auth/oauth-u2m.html)